# RedZone world model pilot 2 (EXP-103, plan A data), on one Colab A100

Proves the whole pipeline end to end: event tokens (vocab v1: possession, stoppage, score, clock) -> a
transformer (~18.5M params: d 384, 8 layers, 6 heads) pretrained on next event + time to the next event -> grades on held-out nights vs count baselines ->
A100 throughput to size Saturday's run. Graded per league (each league split on its own nights). Public inputs only (the kaggle-inputs branch at a pinned commit, play
by play from GitHub: nflverse NFL 2001-24, NBA.com cdn NBA 2020-25 via shufinskiy/nba_data, Retrosheet MLB 1974-2024). No Drive, no secrets, no login. Locked and scorecard nights are never downloaded.

Runtime > Change runtime type > **A100 GPU**, then Runtime > **Run all**. About 1.3 h (~17 CU): data build ~25 min (Retrosheet is ~1,500 small files). If the runtime
restarts but keeps its disk, Run all again: it resumes from the 30-min checkpoint. The last cell prints grades.json
and the CU spent.

In [ ]:
PIN = "d419c5774e88122b365d71675543ed7dae542d16"   # kaggle-inputs commit: factory code pack
REPO = "BouncySlime1215/redzone-everything"
LEAGUES = "nfl,nba,mlb"     # plan A (catalog-OK): nflverse NFL, NBA.com cdn NBA, Retrosheet MLB; hoopR = nba_espn, gated
TRAIN_MIN = 45              # training wall clock; data build ~25 min + grading ~5 min on top
D, LAYERS, HEADS = 384, 8, 6   # ~18.5M params (EXP-103 recipe)
CU_PER_H = {"A100": 13.0, "L4": 5.0, "T4": 2.0}

In [ ]:
import json, os, subprocess, sys, tarfile, time, urllib.request
T_START = time.time()
import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU: Runtime > Change runtime type > A100 GPU, then Run all again")
GPU = torch.cuda.get_device_name(0)
RATE = next((v for k, v in CU_PER_H.items() if k in GPU), None)
print("GPU:", GPU, "| CU/h:", RATE)
RUN = "/content/wmrun"
CODE = f"{RUN}/code"
if not os.path.exists(f"{CODE}/rz/scripts/world_model_pilot.py"):
    os.makedirs(CODE, exist_ok=True)
    for i, ref in enumerate((PIN, PIN[:12], PIN, PIN[:12])):     # the raw CDN can lag a fresh push for a minute
        url = f"https://raw.githubusercontent.com/{REPO}/{ref}/factory/rz_factory_code.tar.gz"
        try:
            print("downloading", url)
            urllib.request.urlretrieve(url, f"{RUN}/code.tar.gz")
            break
        except Exception as e:
            print(" ", type(e).__name__, e)
            time.sleep(30 * i)
    else:
        raise SystemExit("could not download the code pack; Run all again in a minute")
    with tarfile.open(f"{RUN}/code.tar.gz") as t:
        t.extractall(CODE, filter="data")
print("code:", sorted(os.listdir(f"{CODE}/rz")))

In [ ]:
log_path = f"{RUN}/pilot_{time.strftime('%Y%m%d-%H%M')}.log"
with open(log_path, "w") as log:
    p = subprocess.Popen([sys.executable, "-u", f"{CODE}/rz/scripts/world_model_pilot.py", "--out", f"{RUN}/wm",
                          "--leagues", LEAGUES, "--minutes", str(TRAIN_MIN),
                          "--d", str(D), "--layers", str(LAYERS), "--heads", str(HEADS), "--exp", "EXP-103"], cwd=RUN,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in p.stdout:
        print(line, end="")
        log.write(line)
    rc = p.wait()
print("pilot exit code:", rc)

In [ ]:
wall_h = (time.time() - T_START) / 3600
cu = round(wall_h * RATE, 1) if RATE else None
gp = f"{RUN}/wm/grades.json"
if os.path.exists(gp):
    g = json.load(open(gp))
    g["colab"] = {"gpu": GPU, "wall_h": round(wall_h, 2), "cu_spent": cu, "pin": PIN}
    json.dump(g, open(gp, "w"), indent=1)
    print("===== grades.json =====")
    print(json.dumps(g, indent=1))
else:
    print("no grades.json; see", log_path)
print(f"compute: {wall_h:.2f} h on {GPU} = ~{cu} CU")
try:
    from google.colab import files
    files.download(gp)
except Exception as e:
    print("grades.json is at", gp, f"({type(e).__name__})")